# 01 - First look at CONCH

**Goal:** see what CONCH says about real colon tissue, before building the full pipeline.

Run this **on Delta in a GPU Jupyter session**. Run the cells top to bottom (Shift+Enter).

1. Setup and find the data
2. Load CONCH and the tissue prompts
3. Random tissue patches from one TCGA slide -> what does CONCH call them?
4. PathGen patches (which have captions) -> does CONCH agree with the caption?
5. Save the results to files (the Jupyter session will end; files stay)

In [ ]:
# Setup: make `src/` importable and read .env
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config  # must come before the model code (applies HF_HOME)

print("device   :", config.get_device(), "(want 'cuda' on Delta)")
print("HF_TOKEN :", "set" if config.HF_TOKEN else "MISSING - fill in .env")
print("DATA_DIR :", config.DATA_DIR)
print("MODEL_DIR:", config.MODEL_DIR)

## 1. Find the data

This lists what is inside `DATA_DIR` so you can see where the slides and PathGen files live.
Then fill in `SLIDE_DIR` and `PATHGEN_JSON` in the next cell.

In [ ]:
import os

def show_tree(path, depth=2, max_items=15, indent=""):
    path = Path(path)
    try:
        items = sorted(os.listdir(path))
    except Exception as e:
        print(indent + f"(can't read: {e})")
        return
    for name in items[:max_items]:
        p = path / name
        print(indent + name + ("/" if p.is_dir() else ""))
        if p.is_dir() and depth > 1:
            show_tree(p, depth - 1, max_items, indent + "    ")
    if len(items) > max_items:
        print(indent + f"... and {len(items) - max_items} more")

if config.DATA_DIR:
    show_tree(config.DATA_DIR)
else:
    print("Set DATA_DIR in .env (ask organizers for the preloaded data path).")

In [ ]:
# FILL THESE IN after looking at the listing above
SLIDE_DIR = config.DATA_DIR            # folder containing TCGA .svs slides (can be a parent folder)
PATHGEN_JSON = None                    # e.g. config.DATA_DIR / "PathGen" / "PathGen-1.6M.json"

from itertools import islice
slides = sorted(islice(Path(SLIDE_DIR).rglob("*.svs"), 5000)) if SLIDE_DIR else []
print(f"found {len(slides)} .svs slides (stopped looking at 5000)")
for s in slides[:5]:
    print("  ", s.name)

## 2. Load CONCH

First time: downloads ~1-2 GB from Hugging Face (needs `HF_TOKEN` and approved access).
If organizers preloaded the weights, set `CONCH_CHECKPOINT` to that `pytorch_model.bin` file instead.

In [ ]:
from src.models import load_conch, build_text_classifier, classify_images
from src.prompts import CLASSES, PROMPTS_VERSION, build_prompts

CONCH_CHECKPOINT = None   # e.g. config.MODEL_DIR / "conch" / "pytorch_model.bin"

model, preprocess = load_conch(CONCH_CHECKPOINT)
text_emb = build_text_classifier(model)
print("classes:", CLASSES)
print("text classifier shape:", tuple(text_emb.shape), "| prompts", PROMPTS_VERSION)
print("example prompts for 'tumor':", build_prompts()["tumor"][:3])

In [ ]:
# Helper to show patches with CONCH's top guesses
import matplotlib.pyplot as plt
import numpy as np

def top_k(prob_row, k=3):
    order = np.argsort(prob_row)[::-1][:k]
    return [(CLASSES[i], float(prob_row[i])) for i in order]

def show_grid(images, probs, captions=None, ncols=4):
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.6 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, (img, p) in enumerate(zip(images, probs)):
        ax = axes.flat[i]
        ax.imshow(img)
        title = "\n".join(f"{c}: {v:.2f}" for c, v in top_k(p))
        if captions is not None:
            title += "\n" + captions[i][:60] + ("..." if len(captions[i]) > 60 else "")
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()

## 3. One TCGA slide: random tissue patches

Patches are 448 x 448 px at ~0.5 microns/pixel (about 20x), which is CONCH's input size.
**Look at each patch yourself:** does the top label look right? Note which classes it confuses.

In [ ]:
from src.slides import open_slide, get_mpp, get_thumbnail, tissue_mask, read_patch, sample_tissue_coords

SLIDE_PATH = slides[0]      # try others: slides[1], slides[2], ...
slide = open_slide(SLIDE_PATH)
print(SLIDE_PATH.name)
print("size (px):", slide.dimensions, "| microns/pixel:", get_mpp(slide))

thumb = get_thumbnail(slide, 64)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(thumb); axes[0].set_title("thumbnail"); axes[0].axis("off")
axes[1].imshow(tissue_mask(thumb)); axes[1].set_title("tissue mask (yellow = tissue)"); axes[1].axis("off")
plt.show()

In [ ]:
coords = sample_tissue_coords(slide, n=16, seed=0)
patches = [read_patch(slide, x, y) for x, y in coords]
probs, img_emb = classify_images(model, preprocess, patches, text_emb)
show_grid(patches, probs)

## 4. PathGen: patches with captions

PathGen records look like `{wsi_id, position [x, y], caption, file_id}`; each points to a 672 x 672 patch
in a TCGA slide. We only keep records whose slide we found above, read the patch, and compare
CONCH's guess with the caption.

**Uncertain:** we read the 672 px at full resolution. If patches look much more zoomed in/out than the
captions suggest, the PathGen positions may be at a different magnification. Tell us and we'll adjust.

In [ ]:
import json

assert PATHGEN_JSON is not None, "Set PATHGEN_JSON in the 'FILL THESE IN' cell"
with open(PATHGEN_JSON) as f:
    pathgen = json.load(f)
if isinstance(pathgen, dict):          # in case it's {"data": [...]} or similar
    print("top-level keys:", list(pathgen)[:10])
    pathgen = next(v for v in pathgen.values() if isinstance(v, list))
print(f"{len(pathgen):,} records. First one:")
print(json.dumps(pathgen[0], indent=2)[:800])

In [ ]:
# Match PathGen records to the slides we have, using the TCGA barcode before the first "."
slide_by_key = {s.name.split(".")[0]: s for s in slides}
matched = [r for r in pathgen if str(r.get("wsi_id", "")).split(".")[0] in slide_by_key]
print(f"{len(matched):,} PathGen records point at slides in SLIDE_DIR")

rng = np.random.default_rng(0)
picked = [matched[i] for i in rng.choice(len(matched), min(16, len(matched)), replace=False)]

pg_patches, pg_captions = [], []
for r in picked:
    s = open_slide(slide_by_key[str(r["wsi_id"]).split(".")[0]])
    x, y = r["position"]
    pg_patches.append(s.read_region((int(x), int(y)), 0, (672, 672)).convert("RGB"))
    pg_captions.append(r["caption"])

pg_probs, pg_emb = classify_images(model, preprocess, pg_patches, text_emb)
show_grid(pg_patches, pg_probs, captions=pg_captions)

In [ ]:
# Read full captions next to CONCH's guess
for cap, p in zip(pg_captions, pg_probs):
    guess = ", ".join(f"{c} {v:.2f}" for c, v in top_k(p, 2))
    print(f"CONCH: {guess}\nCAPTION: {cap}\n")

## 5. Save results

Saved under `data/results/first_look/` (git-ignored). Download from the Jupyter file browser if you want them
on your laptop. Write down what you noticed (which classes look right/wrong) for the team.

In [ ]:
import pandas as pd

out = config.RESULTS_DIR / "first_look"
out.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASSES])
df.insert(0, "x", [x for x, _ in coords]); df.insert(1, "y", [y for _, y in coords])
df.insert(0, "slide", SLIDE_PATH.name)
df["label"] = [CLASSES[i] for i in probs.argmax(1)]
df.to_csv(out / "random_tiles.csv", index=False)

if "pg_probs" in globals():
    pg = pd.DataFrame(pg_probs, columns=[f"p_{c}" for c in CLASSES])
    pg.insert(0, "caption", pg_captions)
    pg.insert(0, "wsi_id", [r["wsi_id"] for r in picked])
    pg["label"] = [CLASSES[i] for i in pg_probs.argmax(1)]
    pg.to_csv(out / "pathgen_tiles.csv", index=False)

print("saved to", out, "->", sorted(p.name for p in out.iterdir()))